In [ ]:
### Analyze MOFA MODEL Results
### Associate the generated factors with sample meta-data covariates and plot the top features per factor

#############################################
# Prerequisites - Load Libraries

In [ ]:
source('MS1_Functions.r')

In [ ]:
### Inform about start of execution

popup_function_pos('04_Downstream_Factor_Analysis: Execution Started')

In [ ]:
source('MS0_Libraries.r')

In [ ]:
source('MS2_Plot_Config.r')

###############################################
# Preqrequisites Configurations & Parameters

In [ ]:
### Load the parameters that are set via the configuration files

In [ ]:
### Load configurations file
global_configs = read.csv('configurations/Data_Configs.csv', sep = ',')

In [ ]:
head(global_configs,2)

In [ ]:
data_path = global_configs$value[global_configs$parameter == 'data_path']

In [ ]:
data_path

In [ ]:
result_path = global_configs$value[global_configs$parameter == 'result_path']

In [ ]:
result_path

In [ ]:
## Downstream Analysis Configurations (mainly specifying the covariates that should be analyzed in relation to the MOFA factors)

In [ ]:
factor_configs = read.csv( 'configurations/04_Factor_Analysis_Configs.csv', sep = ',')
factor_configs = factor_configs[factor_configs$configuration_name != '',]

In [ ]:
head(factor_configs,2)

In [ ]:
## Get Type color codes from previous script

In [ ]:
type_color_codes = read.csv( 'configurations/03_Type_Color_Codes.csv', sep = ',')

In [ ]:
head(type_color_codes,2)

In [ ]:
type_color_codes$color_code

# Load Data 

## MOFA Input

In [ ]:
#### Load the data based on which the MOFA model was generated (from step 02)

In [ ]:
data_list = list()

In [ ]:
for(i in 1:nrow(factor_configs)){
    # get data name from configuration
    data_name = factor_configs$configuration_name[i]
    
    # Load data
    path = paste0(result_path, '/02_results/02_Combined_Data_',data_name,'_INTEGRATED.csv')
    if(file.exists(path)){
        data_long = read.csv(path)
        data_long$X = NULL
        print(file.info(path)$mtime)
        print(path)

        # save to list
        data_list[[i]] = data_long
        popup_function_pos(paste0('Loaded: ', '02_Combined_Data_',data_name,'_INTEGRATED.csv'))
           }
    else{popup_function_neg(paste0('Error: Cannot Load: ', '02_Combined_Data_',data_name,'_INTEGRATED.csv', ' Please check whether the previous step has been executed successfully'))}
        
    }

In [ ]:
## Example of loaded data

In [ ]:
head(data_long,2)

## MOFA Model

### Load estimated model

In [ ]:
### Load the MOFA model(s) that should be analyzed

In [ ]:
model_list = list()

In [ ]:
for(i in 1:nrow(factor_configs)){
    # get data name from configuration
    mofa_name = factor_configs$mofa_result_name[i]
 
    # Load model
    model_name =  paste0("03_MOFA_MODEL_",mofa_name, '.hdf5')
    outfile = file.path( paste0(result_path, '/03_results/',  model_name) )
    
    if(file.exists(outfile)){
        model = load_model(outfile, verbose = TRUE)

        print(file.info(outfile)$mtime)
        print(outfile)

        # save to list
        model_list[[i]] = model
        popup_function_pos(paste0('Loaded: ', model_name))
        }
    else {  popup_function_neg(paste0('Error: ',result_path, '/03_results/ ', model_name, 'could not be loaded. Check whether the previous steps have been executed successfully'))}
    }
    

### Factor Data

In [ ]:
### Load the factor data to the corresponding MOFA model

In [ ]:
factor_data_list = list()

In [ ]:
for(i in 1:nrow(factor_configs)){
    # get data name from configuration
    mofa_name = factor_configs$mofa_result_name[i]
    
    # load data
    path = paste0(result_path, '/03_results/', '03_Factor_Data_', mofa_name, '.csv')
    
    if(file.exists(path)){
        factors = read.csv(path)
        print(file.info(path)$mtime)
        print(path)

        # Save to list
        factor_data_list[[i]] = factors
        popup_function_pos(paste0('Loaded: ','03_Factor_Data_', mofa_name, '.csv'))
        }
    else{ popup_function_neg(paste0('Error: ','03_Factor_Data_', mofa_name, '.csv', ' could not be loaded. Check whether the previous steps have been executed successfully'))} 
        
        
        
    }
    

In [ ]:
head(factors,2)

### Weight Data

In [ ]:
# Load the feature factor weights to the corresponding MOFA model

In [ ]:
weight_data_list = list()

In [ ]:
for(i in 1:nrow(factor_configs)){
    # get data name from configuration
    mofa_name = factor_configs$mofa_result_name[i]
    
    # laod data
    path = paste0(result_path, '/03_results/', '03_Weight_Data_',  mofa_name, '.csv')
    
    if(file.exists(path)){
        weight_data = read.csv(path)

        print(file.info(path)$mtime)
        print(path)

        # save to list
        weight_data_list[[i]] = weight_data
        popup_function_pos(paste0('Loaded: ','03_Weight_Data_',  mofa_name, '.csv'))}
    
    else{ popup_function_neg(paste0('Error: ','03_Weight_Data_',  mofa_name, '.csv', ' could not be loaded. Check whether the previous steps have been executed successfully and the file exists.'))} 
    
    }
    
    

## Sample Meta Data

In [ ]:
## Load the sample meta-data file that contains covariates that should be associated to factor data

In [ ]:
file_path = paste0(data_path, 'Prepared_Sample_Meta_Data', '.csv')
if(file.exists(file_path)){
    sample_data = read.csv(paste0(data_path, 'Prepared_Sample_Meta_Data', '.csv'))
    sample_data$X = NULL
    popup_function_pos(paste0('Loaded: ','Prepared_Sample_Meta_Data', '.csv'))} else{ popup_function_neg(paste0('Error: ','Prepared_Sample_Meta_Data', '.csv', ' could not be loaded. Please add the file to the specified input data folder.'))}

In [ ]:
### Check whether required columns are in loaded data

In [ ]:
if(sum(strsplit(factor_configs$numeric_covariates , ',')[[1]] %in% colnames(sample_data)) != length(strsplit(factor_configs$numeric_covariates , ',')[[1]])){
     popup_function_neg(paste0('Error: ', strsplit(factor_configs$numeric_covariates , ',')[[1]][!strsplit(factor_configs$numeric_covariates , ',')[[1]] %in% colnames(sample_data)], ' is not included as column within the specified sample dataset.'))}
    

In [ ]:
if(sum(strsplit(factor_configs$categorical_covariates , ',')[[1]] %in% colnames(sample_data)) != length(strsplit(factor_configs$categorical_covariates , ',')[[1]])){
     popup_function_neg(paste0('Error: ', strsplit(factor_configs$categorical_covariates , ',')[[1]][!strsplit(factor_configs$categorical_covariates , ',')[[1]] %in% colnames(sample_data)], ' is not included as column within the specified sample dataset.'))}

In [ ]:
if(! 'sample_id' %in% colnames(sample_data)){
    popup_function_neg(paste0('Error: ', 'columns sample_id is not included in sample data and is required'))}

In [ ]:
str(sample_data$number_of_brain_tumour_sites)

# Downstream Analysis of generated model

# Extract and prepare data for plots

## Merge factors and sample data

In [ ]:
## Combine the factor and sample data table

In [ ]:
factor_data_processed = list()

In [ ]:
for( i in 1:length(factor_data_list)){
    
    ## Transform data to long format
    factors_long = melt(factor_data_list[[i]], id.vars = 'sample_id')
    
    ## Add sample data info
    merged_data_long = merge(factors_long, sample_data, by.x = 'sample_id', by.y = 'sample_id')
    
     ### Filter on relevant factors
     relevant_factors = unlist(str_split(factor_configs$relevant_factors[i], ','))  # get relevant factor from configuration data
     merged_data_long = merged_data_long[merged_data_long$variable %in% relevant_factors,]
    
     factor_data_processed[[i]] = merged_data_long
    }

In [ ]:
head(factor_data_processed[[i]],2)

## Extract explained variance for plotting

In [ ]:
## Get the explained variance information from the MOFA model (these values will be plotted later)

In [ ]:
explained_variance = lapply(model_list, function(x){
    
    # extract variance per factor from model
    data = x@cache$variance_explained$r2_per_factor[[1]]
    
    # extract total variance from model
    total_variance = data.frame( view = rownames(x@cache[["variance_explained"]]$r2_total$group1,2),
                             total_variance = x@cache[["variance_explained"]]$r2_total$group1)
    
    # extract total variance per factor
    total_variance_factor = data.frame(factor = names(rowMeans(x@cache$variance_explained$r2_per_factor[[1]])),
                                   mean_variance = rowMeans(x@cache$variance_explained$r2_per_factor[[1]]))
    
    data = melt(data)
    # merge different variance values
    data = merge(data, total_variance, by.x = 'Var2', by.y = 'view')
    
    })

In [ ]:
head(explained_variance[[1]],2)

## Prepare weight data

In [ ]:
### Adjust the format of the feature factor weight data to long

In [ ]:
feature_weights_list = lapply(weight_data_list, function(x){
    feature_weights_long = melt(x, id.vars = c('variable_name', 'type'))
    
    # adjust formatting of columns
    feature_weights_long$view = feature_weights_long$type
    feature_weights_long$gene = str_replace(feature_weights_long$variable_name, '.*__', '')
    
    return(feature_weights_long)
    })
    

In [ ]:
head(feature_weights_list[[1]],2)

## Get top features per factor and amounts for diff thresholds

In [ ]:
## Get the x% of top features per factor based on the specified threshold (x) in the configuration file

In [ ]:
geneset_oi_amounts_list = list()

In [ ]:
for(i in 1:length(feature_weights_list)){
    
    feature_weights_long = feature_weights_list[[i]]
    
    # get threshold for top variables from configuration
    top_variable_perc = factor_configs$top_variable_thres[i]
    
    
    ## Define amount of top genes per fraction 
    geneset_oi_pos_per_factor_analyze = feature_weights_long %>% group_by(variable) %>% dplyr::arrange( desc(value),  .by_group = TRUE)  %>% top_frac(  as.numeric(top_variable_perc), value)
    geneset_oi_pos_per_factor_analyze$direction = 'positive'
    
    geneset_oi_neg_per_factor_analyze = feature_weights_long %>% group_by(variable) %>% dplyr::arrange(desc(value),  .by_group = TRUE)  %>% top_frac( - as.numeric(top_variable_perc), value)
    geneset_oi_neg_per_factor_analyze$direction = 'negative'
    
    geneset_oi_analyze = rbind(geneset_oi_pos_per_factor_analyze, geneset_oi_neg_per_factor_analyze)
    geneset_oi_analyze$fraction =  as.numeric(top_variable_perc)

    
    ## Calculate the amount of top features per type
    dimensions = unique(feature_weights_long[,c('view', 'variable')])
    
    amount_geneset_oi_type = geneset_oi_analyze %>% group_by(type, view, variable) %>% dplyr::count()
    amount_geneset_oi_type = merge(dimensions, amount_geneset_oi_type, all.x = TRUE) # to avoid missing dimensions
    amount_geneset_oi_type$fraction = as.numeric(top_variable_perc)
    
    geneset_oi_amounts = amount_geneset_oi_type
    
    ## Calculate the total amount of features per type
    features_per_type = feature_weights_long %>% group_by(type, view, variable) %>% dplyr::count()
    colnames(features_per_type) = c('type', 'view', 'variable', 'amount_features')
    
    ## Merge and calculate percentage
    geneset_oi_amounts = merge(  geneset_oi_amounts,features_per_type, all.x = TRUE)
    geneset_oi_amounts$percentage = geneset_oi_amounts$n / geneset_oi_amounts$amount_features
    
    ## Adjust NA's
    geneset_oi_amounts[is.na(geneset_oi_amounts)] = 0 # NA when there are no features for this dimension among top %
    
    ## save for the threshold/ config
    geneset_oi_amounts_list[[i]] = geneset_oi_amounts
    }
    
    
    
    

In [ ]:
head(    geneset_oi_amounts_list[[i]],2)

# Plots

In [ ]:
### Generate the plots to evaluate the factor values in relation to different sample-meta data covariates

## Investigate relationship of factors with numeric values

In [ ]:
### Calculate correlatins with numeric features

In [ ]:
head(factor_data_processed[[1]],2)

In [ ]:
figure_name = "FIG04_Factor_Association_with_numeric_features_"  # Define the name of the plot

In [ ]:
# Set the sizes of the plot
width_par = 8.07
height_par = 3.5

In [ ]:
factor_configs

In [ ]:
for(j in 1:length(factor_data_processed)){
    ## get the variables
    numeric_variables = unlist(str_split(factor_configs$numeric_covariates[j], ','))
    
    
    ## calculate correlations
    cor_plot = list()
    
    ## get the data
    merged_data_long = factor_data_processed[[j]]
    
    ### calculate correlations
    for(i in numeric_variables){
        cor_plot[[i]] = ggplot(merged_data_long, aes(x = value, y = get(i))) + facet_wrap(.~ variable, scale = 'free') +
        geom_point(size = 0.2) + plot_config + geom_smooth(method='lm', col = 'blue3', se = FALSE) + stat_cor(method = 'pearson') + ylab(i)


        }
    
    
    ### Plot the scatterplots
    
    # get the name for saving
    mofa_name = factor_configs$mofa_result_name[j]
    
    pdf(paste0('figures/04_figures/', figure_name,mofa_name, '.pdf'), width =width_par, height =height_par, onefile = TRUE)
    for (i in names(cor_plot)) {
      print(cor_plot[[i]])  
    }
    dev.off()
    }
    
    

## Investigate relationship with categorical values

In [ ]:
### Generate boxplots to evaluate the factor value difference for categorical sample meta data variables

In [ ]:
# Specific Text Descriptions:
xlabel = xlab('') 
ylabel = ylab('Factor Value')

In [ ]:
# Specify Figure Name
figure_name = paste0("FIG04_Factor_Association_with_categorical_features_")

In [ ]:
# Specify sizes of the plot
width_par = 8.07
height_par = 3

In [ ]:
library(ggplot2)
library(dplyr)
library(reshape2)   # for melt if needed
library(stringr)

for(j in 1:length(factor_data_processed)){
  
  # Get the categorical variable names for this dataset
  categorical_variables <- unlist(str_split(factor_configs$categorical_covariates[j], ','))
  
  # Get the processed (melted and merged) data
  merged_data_long <- factor_data_processed[[j]]
  
  # Rename the melted column "variable" to "feature" to avoid conflicts
  names(merged_data_long)[names(merged_data_long) == "variable"] <- "feature"
  
  mofa_name <- factor_configs$mofa_result_name[j]
  
  pdf(paste0('figures/04_figures/', figure_name, mofa_name, '.pdf'),
      width = width_par, height = height_par)
  
  for(cat_var in categorical_variables){
    
    # Create a new column "condition" based on the current categorical variable
    merged_data_long$condition <- merged_data_long[, cat_var]
    
    # Filter out rows where condition is NA, "NaN", or an empty string
    vis_data <- merged_data_long[!(is.na(merged_data_long$condition) | 
                                    merged_data_long$condition == "NaN" | 
                                    merged_data_long$condition == ""), ]
    
    # Ensure the condition is a factor and value is numeric:
    vis_data$condition <- factor(vis_data$condition)
    vis_data$value <- as.numeric(vis_data$value)
    
    # Remove features that only have one condition (i.e. one box per feature)
    valid_features <- vis_data %>%
      group_by(feature) %>%
      summarise(n_levels = n_distinct(condition)) %>%
      filter(n_levels >= 2) %>%
      pull(feature)
    
    # If no feature has at least 2 levels, skip this category
    if(length(valid_features) == 0) next
    
    # Keep only valid features
    vis_data <- vis_data %>% filter(feature %in% valid_features)
    
    pvals_df <- vis_data %>%
      group_by(feature) %>%
      group_modify(~ {
        # Check if there are at least two distinct groups AND that each group has at least 2 observations
        if(n_distinct(.x$condition) < 2 || min(table(.x$condition)) < 2) {
          return(tibble(p_value = NA_real_, y.position = NA_real_))
        } else if(n_distinct(.x$condition) == 2) {
          p_val <- t.test(value ~ condition, data = .x)$p.value
        } else {
          p_val <- summary(aov(value ~ condition, data = .x))[[1]]$`Pr(>F)`[1]
        }
        tibble(p_value = p_val, y.position = max(.x$value, na.rm = TRUE) * 1.05)
      }) %>%
      ungroup() %>%
      filter(!is.na(p_value))
    
    # Format the p-value with 3 digits and add as a label
    pvals_df <- pvals_df %>% mutate(p_label = sprintf("p = %.3f", p_value))
    
    # Ensure that the 'feature' variable in pvals_df is a factor with the same levels as in vis_data
    pvals_df$feature <- factor(pvals_df$feature, levels = unique(vis_data$feature))
    
    # Use the appropriate color scale from colors_list if available
    color_scale <- if(cat_var %in% names(colors_list)) {
      colors_list[[cat_var]]
    } else {
      colors_list[['default']]
    }
    
    # Build the ggplot; facets are based on "feature"
    g <- ggplot(vis_data, aes(x = condition, y = value, col = condition)) +
      facet_grid(. ~ feature) +
      plot_config +
      xlabel +
      ylabel +
      ggtitle(cat_var) +
      theme(legend.position = "bottom", axis.text.x = element_blank()) +
      geom_boxplot(outlier.size = 0.05) +
      geom_point(size = 0.5) +
      color_scale +
      # Annotate each facet with its own computed p-value.
      geom_text(
        data = pvals_df,
        mapping = aes(x = Inf, y = y.position, label = p_label),
        inherit.aes = FALSE,
        hjust = 1.1,
        vjust = 1,
        size = 3  # smaller text size
      )
    
    print(g)
  }
  
  dev.off()
}

## Feature Overview per Factor

In [ ]:
## Generate the overview of the amount of top x% of features per factor and view (in relation to total amount of features per view)

In [ ]:
## Plot the heatmap showing the explained variance of the factor per view

In [ ]:
variance_plots = list()

In [ ]:
for(j in 1:nrow(factor_configs)){

    ## get the relevant factor and top variable fraction
    factor_var = unlist(str_split(factor_configs$relevant_factors[j], ','))
    
    
    ## Explained Variance Heatmap Plot (for each factor)
    explained_variance_heatmap = list()
    for(i in factor_var ){
        data = explained_variance[[j]]   # get prepared variance data
        data$Var2 = as.character(data$Var2)
        data$Var2 = factor(data$Var2, levels = sort(unique(data$Var2)))  # recode to ensure right ordering
        
        data_plot = data[data$Var1 == i,]
        data_plot$Var1 = 'Explained'

        explained_variance_heatmap[[i]] = ggplot() + scale_fill_gradient(low="white", high="black") + 
        ylabel + xlabel + plot_config_heatmap +  theme(axis.text.y = element_text(hjust = 0, vjust = 0.5)) +
        geom_tile(data = data_plot, mapping = aes(Var1,  Var2, fill= value)
                 )  +
        ggtitle(i)
        }
    
    variance_plots[[j]] =  explained_variance_heatmap
    }

In [ ]:
#variance_plots[[1]]

In [ ]:
## Generate the barplots with showing the amount of top x% of features

In [ ]:
barplot_top_features_percentages = list()
barplot_top_features_absolute = list()

In [ ]:
for(j in 1:nrow(factor_configs)){
    
    # get configurations
    top_var_fraction = factor_configs$top_variable_thres[j]
    factor_var = unlist(str_split(factor_configs$relevant_factors[j], ','))
    
    ## Barplots with top features per factor
    
    ## 1: Percentages (amount of top x% of features in relation to total amount)
    xlabel = xlab('View') 
    ylabel = ylab('Percentage of total features')
    
    percentage_plot_1_perc = list()
    
    for(i in factor_var ){

            geneset_oi_amounts = geneset_oi_amounts_list[[j]]
            geneset_oi_amounts$view  = as.character(geneset_oi_amounts$view)
            geneset_oi_amounts$view = factor(geneset_oi_amounts$view, levels = sort(unique(geneset_oi_amounts$view))) # recode to ensure right ordering

            percentage_plot_1_perc[[i]] = ggplot(data = geneset_oi_amounts[(geneset_oi_amounts$variable == i),], aes(x = view, y = percentage*100, fill = view)) +
            xlabel + 
            ylabel + 
            plot_config + 
            geom_bar(stat="identity") + coord_flip() + theme(legend.position = 'none') +
            ggtitle(paste0('Top ', 2*as.numeric( top_var_fraction) *100, '% of features')) +
            geom_hline(yintercept = 2*as.numeric( top_var_fraction)*100, 
                    color = "black", size=1) + scale_fill_manual(values = type_color_codes$color_code)
        
    }
    barplot_top_features_percentages[[j]] =  percentage_plot_1_perc 
    
    
    
    ## 2: Absolute Values (absolute amount of top x% of features of the view)
    
    xlabel = xlab('View') 
    ylabel = ylab('Amount features')
    
    absolute_plot_1_perc = list()
    
    # one selected threshold + absolute amount
    for(i in factor_var ){
            geneset_oi_amounts = geneset_oi_amounts_list[[j]]
            geneset_oi_amounts$view = as.character(geneset_oi_amounts$view)
            geneset_oi_amounts$view = factor(geneset_oi_amounts$view, levels = sort(unique(geneset_oi_amounts$view)))# recode to ensure right ordering

            absolute_plot_1_perc[[i]] = ggplot(data = geneset_oi_amounts[(geneset_oi_amounts$variable == i),], aes(x = view, y = n, fill = view)) +
            xlabel + 
            ylabel + 
            plot_config + 
            geom_bar(stat="identity") + coord_flip()  + theme(legend.position = 'none')+ 
            ggtitle(paste0('Top ', 2*as.numeric( top_var_fraction) *100, '% of features')) + scale_fill_manual(values = type_color_codes$color_code) # TBD: maybe improve even with default value + specifying all colors via table
        }
    
    barplot_top_features_absolute[[j]] = absolute_plot_1_perc
       

}  

In [ ]:
## Combine the plots and save

In [ ]:
# Specify the figure name
figure_name = paste0( "TEST_FIG04_Top_Feature_Overview_per_Factor")

In [ ]:
# Specify the sizes of the plot
width_par = 8.07
height_par =2.8

In [ ]:
factor_configs

In [ ]:
for(j in 1:nrow(factor_configs)){
    mofa_name = factor_configs$mofa_result_name[j]
    
    # get the relvant plot
    explained_variance_heatmap = variance_plots[[j]]
    absolute_plot_1_perc = barplot_top_features_absolute[[j]]
    percentage_plot_1_perc = barplot_top_features_percentages[[j]]


    pdf(paste0('figures/04_figures/', figure_name, '_',   mofa_name, '.pdf'), width =width_par, height =height_par)
    for( i in 1:length(explained_variance_heatmap)){
        legend = get_legend(explained_variance_heatmap[[i]])

        combined1 = ggarrange(explained_variance_heatmap[[i]] + theme(legend.position = 'none'),
                             absolute_plot_1_perc[[i]] + theme(axis.text.y = element_blank(),axis.ticks.y = element_blank(),axis.title.y = element_blank() ), 
                             percentage_plot_1_perc[[i]] + theme(axis.text.y = element_blank(),axis.ticks.y = element_blank(),axis.title.y = element_blank() ),  
                             nrow=1, widths = c(2.2,1,1))
        combined1 = annotate_figure(combined1, right = legend)

        print( combined1)
        }
    dev.off()   
    }

In [ ]:
### Inform about finalkzation of execution

popup_function_pos('04_Downstream_Factor_Analysis: Execution Finished')

In [ ]:
Sys.sleep(20)
popup_function_info('04_Downstream_Factor_Analysis')

In [ ]:

figure_name <- "TEST2_FIG04_Top_Feature_Overview_per_Factor"
width_par   <- 8.07
height_par  <- 2.8
color_palette <- c(
  'OPC'             = '#1f77b4',
  'OPC.like'        = '#4b9cd3',
  'Oligodendrocyte' = '#6baed6',
  'Astrocyte'       = '#9ecae1',
  'TRAIL.Astrocytes'= '#ff00ff',
  'AC.like'         = '#6a51a3',
  'NPC.like'        = '#8c6bb1',
  'RG'              = '#bcbddc',
  'Neuron'          = '#dadaeb',
  'TAM.MG'          = '#d62728',
  'TAM.BDM'         = '#e66767',
  'E.MDSCs'         = '#8B4513',
  'M.MDSCs'         = '#DAA520',
  'Mono'            = '#fdae6b',
  'Neutrophil'      = '#fdd0a2',
  'DC'              = '#e31a1c',
  'CD4.CD8'         = '#fc4e2a',
  'NK'              = '#feb24c',
  'B.cell'          = '#ffeda0',
  'Plasma.B'        = '#f768a1',
  'Mast'            = '#ae017e',
  'Endothelial'     = '#31a354',
  'Mural.cell'      = '#74c476',
  'MES.like'        = '#bae4b3',
  # Adding colors for the special views
  'bulk_5000hvg_counts_symbols'    = '#333333', 
  'celltype_proportions_zComp_CLR' = '#999999'
)
# Renaming mapping for labels
rename_map <- c(
  "bulk_5000hvg_counts_symbols"    = "RNAseq",
  "celltype_proportions_zComp_CLR" = "Cell Type Proportions"
)

# Logical order for views
view_order <- c(
  "OPC", "Oligodendrocyte", "Astrocyte", "Neuron",
  "TAM.MG", "TAM.BDM", "E.MDSCs", "M.MDSCs", "Mono", "DC", "CD4.CD8", "NK", "Mast",
  "Endothelial", "Mural.cell",
  "OPC.like", "AC.like", "NPC.like", "MES.like",
  "celltype_proportions_zComp_CLR", 
  "bulk_5000hvg_counts_symbols"
)

# Helper function for label cleaning
clean_view_names <- function(x) {
  x <- as.character(x)
  if (x %in% names(rename_map)) {
    return(rename_map[x])
  } else {
    return(gsub("\\.", "-", x)) # TAM.MG -> TAM-MG
  }
}

# Define the labeled order (reversed for ggplot coord_flip)
labeled_order <- sapply(rev(view_order), clean_view_names)


for(j in 1:nrow(factor_configs)){
  
  mofa_name <- factor_configs$mofa_result_name[j]
  factor_var <- unlist(str_split(factor_configs$relevant_factors[j], ','))
  top_var_fraction <- as.numeric(factor_configs$top_variable_thres[j])
  
  # Load specific dataset for this config
  data_variance <- explained_variance[[j]]
  geneset_oi_amounts <- geneset_oi_amounts_list[[j]]
  
  # Prepare output folder
  dir.create("figures/04_figures/", recursive = TRUE, showWarnings = FALSE)
  pdf(paste0('figures/04_figures/', figure_name, '_', mofa_name, '.pdf'), 
      width = width_par, height = height_par)
  
  for(i in factor_var){
    
    
    
    # Heatmap Data
    data_heat <- data_variance[data_variance$Var1 == i, ]
    data_heat$Var1 <- 'Explained'
    data_heat$Var2_labeled <- factor(sapply(data_heat$Var2, clean_view_names), levels = labeled_order)
    
    # Barplot Data
    data_bar <- geneset_oi_amounts[geneset_oi_amounts$variable == i, ]
    data_bar$Var2_labeled <- factor(sapply(data_bar$view, clean_view_names), levels = labeled_order)
    
   
    
    # Heatmap Panel
    p_heat <- ggplot(data_heat, aes(x = Var1, y = Var2_labeled, fill = value)) +
      geom_tile() +
      scale_fill_gradient(low = "white", high = "black", name = "Var.") +
      xlab("Explained") + ylab("View") +
      theme_minimal() +
      theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 12),
          axis.text.y = element_text(hjust = 0, vjust = 0.5, size = 8),
            axis.text.x = element_blank(),
            panel.grid = element_blank()) +
      ggtitle(i)

    # Absolute Barplot
    p_abs <- ggplot(data_bar, aes(x = Var2_labeled, y = n, fill = view)) +
      geom_bar(stat = "identity") +
      coord_flip() +
      scale_fill_manual(values = color_palette) +
      xlab("") + ylab("Amount features") +
      theme_minimal() +
      theme(legend.position = 'none',
            axis.text.y = element_blank(),
            axis.ticks.y = element_blank(),
            panel.grid.minor = element_blank(),
            panel.grid.major.y = element_blank()) +
      ggtitle(paste0('Top ', 2 * top_var_fraction * 100, '%'))

    # Percentage Barplot
    p_perc <- ggplot(data_bar, aes(x = Var2_labeled, y = percentage * 100, fill = view)) +
      geom_bar(stat = "identity") +
      geom_hline(yintercept = 2 * top_var_fraction * 100, color = "black", linetype = "dashed") +
      coord_flip() +
      scale_fill_manual(values = color_palette) +
      xlab("") + ylab("Percentage of total features") +
      theme_minimal() +
      theme(legend.position = 'none',
            axis.text.y = element_blank(),
            axis.ticks.y = element_blank(),
            panel.grid.minor = element_blank(),
            panel.grid.major.y = element_blank()) +
      ggtitle(paste0('Top ', 2 * top_var_fraction * 100, '%'))

    #--- Stitch Panels ---
    
    legend <- get_legend(p_heat)
    combined <- ggarrange(
      p_heat + theme(legend.position = 'none'),
      p_abs,
      p_perc,
      nrow = 1, widths = c(2.2, 1, 1), align = "h"
    )
    
    final_plot <- annotate_figure(combined, right = legend)
    print(final_plot)
  }
  
  dev.off()
  message("Saved: ", mofa_name)
}